# SFINCS — NJ Sandy: premier results viewer

Opens the adopted premier **read-only** and plots it. `faber-waves-premier` is the 2026-07-14
rebuild that plugged the Navesink leak and carved open Shark River Inlet (two DEM/mask defects,
not physics; full write-up in `reports/shrewsbury_investigation.md`).

Everything here is that one run: its build, its forcing, its skill against the observations, and
finally the storm animated hour by hour. The superseded broken-domain runs are not shown — for
the before/after argument that retired them, see
`archive/notebooks/sfincs-nj-sandy-viz-estuary-leakfix.ipynb`.

## Setup

In [ ]:
# Viz stack (this import also primes PROJ before hydromt loads).
import sys
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))
from hydromt_sfincs import SfincsModel
from nj_sfincs import animate, plots, validate

# The run to view (methodology + wave field come from this one). Adopted premier:
EXP = "faber-waves-premier"
exp_dir = ROOT / "experiments" / EXP
if not exp_dir.exists():
    exp_dir = ROOT / "model"  # fall back to the reference build
    print(f"experiments/{EXP} not found — showing the reference model/ build")
print("viewing:", exp_dir)


In [ ]:
# Open read-only: `sf` (build + forcing inputs) and `mod` (solver output).
# load_floodmap reuses the cached floodmap_hmax_lev3.tif when it is newer than
# sfincs_map.nc, so this is ~30 s on a run already downscaled and ~2 min on a fresh
# one. Pass force=True if you suspect the cache.
sf = SfincsModel(
    str(exp_dir), data_libs=[str(ROOT / "data" / "data_catalog.yml")], mode="r"
)
sf.read()
mod, da_hmax, da_dep = validate.load_floodmap(exp_dir)
print("output vars:", list(mod.output.data.keys()))


## Methodology — the build

The sealed run's own grid, topobathy, and mask.

### Quadtree grid

In [ ]:
plots.plot_grid(sf);

### Topobathy (interactive) — pan/zoom the dunes, inlets, dredged channels

In [ ]:
plots.plot_topobathy(sf)

### Mask — active interior, water-level boundary, outflow

In [ ]:
plots.plot_mask(sf);

## Methodology — the forcing

The compound drivers, read back from the written model.

### Surge boundary (NOAA CO-OPS)

In [ ]:
plots.plot_surge(sf);

### Wind + pressure (ERA5)

In [ ]:
plots.plot_wind_pressure(sf);

### Rainfall (NOAA AORC)

In [ ]:
plots.plot_rain(sf);

### River discharge (USGS)

In [ ]:
plots.plot_discharge(sf);

## Results — how the premier scores

Skill of `faber-waves-premier` against the observations: the wave field, the flood extent, the
gauges, the USGS high-water marks, and the FEMA MOTF footprint.

In [ ]:
# The panel helpers all take {label: experiment_dir}, so a single-entry dict
# renders one panel. Add runs here to compare against something.
RUNS = {"faber-waves-premier (premier)": EXP}

### Wave field — SnapWave Hm0 at peak

The lee behind Sandy Hook: the hook shelters the bay from the swell.

In [ ]:
res = plots.plot_wave_field_panels(RUNS)
if res is None:
    print("no wave output for this run (waves off, or no hm0)")

### Maximum flood depth — Shrewsbury / Navesink estuary

The estuary the leak used to drain. Sealed, the back-bays fill. Red circles are USGS
high-water-mark locations (quality ≤ 2).

In [ ]:
plots.plot_engine_panels(RUNS);

### Gauge & pre-storm tide

Read the **pre-storm tide** (left of the dotted "gauge dies" line): a working tide floods *and*
ebbs. Shark oscillates at last now that the inlet is carved open — obs range 1.54 m, modelled
1.36 m, rising 0.46 of the time.

In [ ]:
plots.plot_gauge_verification(RUNS);

### USGS high-water-mark residuals

Model − obs: red = too high, blue = too low, ✕ = dry where a mark says wet. 19 marks scored,
bias +0.32 m, RMSE 0.48 m, none dry.

In [ ]:
plots.plot_hwm_residual_panels(RUNS);

### FEMA MOTF flood extent

**Read the CSI, not the POD** — MOTF is a bathtub surface that shares provenance with our own
high-water marks, and its POD structurally rewards over-flooding: flood everything and POD is
perfect. Treat it as an extent *consistency* check, not an independent observation.

The premier scores **CSI 0.71 / POD 0.80 / FAR 0.14** — against 0.51 / 0.56 / 0.17 on the old
broken domain. (The sealed run with waves *off* reaches CSI 0.64, so most of the gain is the
domain fix, not the waves.)

In [ ]:
plots.plot_motf_panels(RUNS);

## The decision — why this run

`faber-waves-premier` is the **adopted premier**: gauge within 0.10 m of the surveyed Shrewsbury
crest (2.84 vs 2.935 m), Shark tide alive, best MOTF. **Faber over Galibier** — identical without
waves, but Galibier+waves overshoots hard (gauge +0.57 m, HWM bias +0.97), so Galibier is
unofficially retired.

⚠️ **Locality caveat:** the open coast that never broke drifted ~0.1 m on the rebuild
(south_coast −0.055 → +0.048), so the fix is not *purely* local — small next to the gains,
but worth a look.

In [ ]:
# The four sealed candidates (tide, gauge, per-basin HWM bias). The broken-domain
# reference rows in the CSV are filtered out — this is the sealed 2x2 only.
# Produced by scripts/analyze_sealed.py; re-run that script if the CSV is stale.
sealed_csv = ROOT / "reports" / "sealed_premier.csv"
if sealed_csv.exists():
    sealed = pd.read_csv(sealed_csv)
    sealed = sealed[sealed["run"].str.startswith("sealed_")]
    cols = [c for c in ["desc", "shark_frac_rising", "shark_tide", "shrews_tide",
                        "gauge", "gauge_err", "shrewsbury_navesink", "shark_river",
                        "south_coast", "atlantic_oceanfront", "rmse"] if c in sealed]
    display(sealed[cols].round(3))
    print("\nADOPTED: faber-waves-premier (Faber, waves on).")
    print("Locality caveat: south_coast drifted -0.055 -> +0.048 on the rebuild.")
else:
    print("run:  NJ_ROOT=$PWD PYTHONPATH=$PWD python scripts/analyze_sealed.py")

---

## The storm, hour by hour

Everything above is a single moment — a max, a peak, a before/after. This section plays the
run: 73 hourly frames from 2012-10-28 00:00 to 10-31 00:00 UTC.

**Why this is fast.** The quadtree is fixed for a run; only the face *values* change with
time. So `nj_sfincs.animate` rasterizes the face **indices** once (~0.6 s) and every frame
after that is an array lookup — a full 73-frame stack builds in about a second, against the
minutes a per-frame render of 547k mesh polygons would cost. See the module docstring.

### Flood depth — the Shrewsbury / Navesink estuary

Watch the barrier island at Sea Bright go under around 10-29 20:00, then the estuary behind it
fill. The permanently-wet channel and shelf (bed below −0.5 m) are masked out so the eye follows
the **new** flooding rather than water that was always there; pass `mask_ocean=False` to keep it.

Depth is capped at 3 m to match `plot_engine_panels` — a percentile scale would stretch to the
15–28 m of open ocean and render the actual inundation near-white.

In [ ]:
from IPython.display import HTML

anim = animate.animate_field(EXP, "depth", window="shrewsbury", fps=6)
HTML(anim.to_jshtml())

### SnapWave Hm0 — the lee behind Sandy Hook

The open coast lights up as Sandy closes in while the water **behind** the hook stays dark: the
hook shelters Sandy Hook Bay from the swell, and SnapWave cannot diffract energy into that lee
(see `project_bay_waves_plan`).

The bright band on the right edge is the SnapWave **inflow boundary** (~8 m of imposed offshore
Hm0) — real forcing, not an artifact. SnapWave runs on a sub-domain of the hydro mesh, which is
why the field simply stops out there.

In [ ]:
anim = animate.animate_field(EXP, "hm0", window="sandy_hook", fps=6)
HTML(anim.to_jshtml())

### Shark River — the inlet that used to be a dam

On the old domain the lidar had paved Shark River Inlet shut (sill +0.57 m, above MSL) and the
estuary sat at exactly +0.00 m through the entire storm. Post-carve it breathes: this animation
shows the tide running in and out of the inlet before the surge arrives.

In [ ]:
anim = animate.animate_field(EXP, "depth", window="shark", fps=6)
HTML(anim.to_jshtml())

### Interactive — pan, zoom, scrub

Same index raster underneath, so the time slider is instant (the whole series is already in
memory; it is not re-reading the map file per frame). Hover reads the value out; the backdrop is
Esri satellite imagery.

Swap `var` for any key of `animate.FIELDS` (`depth`, `zs`, `hm0`, `hm0ig`, `tp`) and `window`
for any key of `animate.WINDOWS` (`domain`, `shrewsbury`, `sandy_hook`, `shark`).

In [ ]:
animate.explore_field(EXP, var="tp", window="shrewsbury")

### Saving a GIF

`to_jshtml` embeds every frame as base64 PNG, which is fine inline but makes for a fat notebook
(~20 MB for the depth loop). To hand someone a file instead, write a GIF — pillow is the writer,
no ffmpeg needed (the env has none).

In [ ]:
out = ROOT / "reports" / "sandy_depth_shrewsbury.gif"
anim = animate.animate_field(EXP, "depth", window="shrewsbury", every=1)
anim.save(out, writer="pillow", fps=6, dpi=90)
print("wrote", out, f"({out.stat().st_size/1e6:.1f} MB)")

- Ask Jimmy tide/surge dataset and connect me with spanish surfer
- Ask DaNa to connect to NOS